# Forecasting Model: Lag12 + Load Factor

Notebook 15a demonstrated that `delay_rate_lag12` (same month last year) significantly improves the forecasting model. Regression models are particularly improved: Ridge R² +0.062 (0.430→0.492), RF R² +0.045 (0.462→0.506) and while the XGB classification model is only marginally better F1 +0.006 (0.731→0.737).

The hypothesis is that in the case of forecasting (where same-month weather features are unavailable), `load_factor_lag1` (previous month's load factor) may provide additional predictive signal. Load factor captures demand-side pressure that is not represented by other lagged features.

Only `load_factor_lag1` is tested, as it is strictly available at forecast time. Same-month load factor and interaction terms with weather are excluded since they would not be known when making predictions.

## Reference performance (notebook 15a, forecasting with lag12)

| Model | R² | F1 |
|-------|----|----|   
| Ridge | 0.492 | - |
| Random Forest | 0.506 | - |
| XGBoost | - | 0.737 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, f1_score

np.random.seed(42)

try:
    import xgboost as xgb
    HAS_XGB = True
    print('XGBoost available')
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

%matplotlib inline

## 1. Data Preparation

The training data is loaded following notebook 15a. Load factor is sourced from the BITRE Monthly Airline Performance dataset (one industry-aggregate value per month) and merged by `year_month`.

In [ ]:
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print('Shape: {}'.format(df.shape))
print('Date range: {} to {}'.format(df['year_month'].min(), df['year_month'].max()))

In [ ]:
# Load BITRE Monthly Airline Performance and extract load factor
activity_candidates = [
    '../data/raw/monthly-airline-performance-november-2025.xlsx',
    '../data/raw/monthly-airline-performance-october-2025.xlsx',
]
ACTIVITY_FILE = next((f for f in activity_candidates if os.path.exists(f)), None)
if ACTIVITY_FILE is None:
    raise FileNotFoundError('Monthly airline performance file not found.')
SHEET_NAME = 'Domestic airlines'

df_activity = pd.read_excel(ACTIVITY_FILE, sheet_name=SHEET_NAME, header=None, skiprows=8)
df_activity.columns = [
    'year', 'month_name', 'hours_flown', 'aircraft_km_flown_000', 'aircraft_departures',
    'total_rev_pax_ud', 'freight_tonnes_ud', 'mail_tonnes_ud',
    'total_rev_pax_tob', 'total_rev_pax_tob_inc_intl',
    'freight_tonnes_tob', 'mail_tonnes_tob',
    'total_rpk_000', 'pax_tonne_km_000', 'freight_tonne_km_000',
    'mail_tonne_km_000', 'total_tonne_km_000',
    'available_seat_km_000', 'available_tonne_km_000', 'available_seats_000',
    'pax_load_factor_pct', 'weight_load_factor_pct',
    'total_charter_pax_tob', 'charter_aircraft_departures'
]

df_activity['year_numeric'] = pd.to_numeric(df_activity['year'], errors='coerce')
valid_months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'June',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

df_act = df_activity[
    df_activity['year_numeric'].notna() &
    df_activity['month_name'].isin(valid_months)
].copy()

df_act['year'] = df_act['year_numeric'].astype(int)
df_act = df_act.drop(columns=['year_numeric'])
df_act['month_name'] = df_act['month_name'].replace({'June': 'Jun'})

month_map = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
             'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
df_act['month_num'] = df_act['month_name'].map(month_map)
df_act['year_month'] = df_act['year'].astype(str) + '-' + df_act['month_num'].astype(str).str.zfill(2)
df_act['load_factor'] = pd.to_numeric(df_act['pax_load_factor_pct'], errors='coerce') / 100.0

# Exclude COVID period
covid_mask = (df_act['year'] == 2020) & (df_act['month_num'] >= 4)
df_act = df_act[~covid_mask].copy()
df_act = df_act[df_act['year'] >= 2009].copy()

df_lf = df_act[['year_month', 'load_factor']].groupby('year_month', as_index=False).mean()
df = df.merge(df_lf, on='year_month', how='left')

matched = df['load_factor'].notna().sum()
match_rate = matched / len(df) * 100

print('Load factor source: {}'.format(ACTIVITY_FILE))
print('Load factor records: {}'.format(len(df_lf)))
print('Match rate: {:.1f}% ({}/{})'.format(match_rate, matched, len(df)))

In [ ]:
# Filter low-volume airline-routes and anomalous routes (matching 15a)
volume_threshold = 40
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean()
low_volume = airline_route_volume[airline_route_volume < volume_threshold]
high_volume_ar = airline_route_volume[airline_route_volume >= volume_threshold].index.tolist()
df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()

anomalous_routes = ['Melbourne_Hobart', 'Adelaide_Sydney', 'Perth_Brisbane']
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()

print('Volume threshold: {} flights/month'.format(volume_threshold))
print('Excluded low-volume: {}'.format(len(low_volume)))
print('Excluded anomalous routes: {}'.format(anomalous_routes))
print('Records after filtering: {:,}'.format(len(df_filtered)))

## 2. Feature Engineering

Standard lag features and encoding are created following notebook 15a. No same-month weather features are included (unavailable at forecast time). The only addition is `load_factor_lag1` (previous month's industry-aggregate load factor).

In [ ]:
# Standard lag features (same as 15a)
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']
df_filtered['delay_rate_lag12'] = df_filtered.groupby('airline_route')['delay_rate'].shift(12)

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = sorted(airline_dummies.columns.tolist())

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = sorted(route_dummies.columns.tolist())

print('Airlines: {}'.format(len(airline_cols)))
print('Routes:   {}'.format(len(route_cols)))

In [ ]:
# Load factor lag1 (previous month, available at forecast time)
df_filtered['load_factor_lag1'] = df_filtered.groupby('airline_route')['load_factor'].shift(1)

valid_lf = df_filtered['load_factor_lag1'].notna().sum()
print('load_factor_lag1 valid: {:,} / {:,}'.format(valid_lf, len(df_filtered)))

In [ ]:
# Correlation analysis
corr_cols = ['delay_rate', 'delay_rate_lag1', 'delay_rate_lag12',
             'delay_rate_gradient', 'sectors_scheduled', 'load_factor_lag1']
valid_data = df_filtered[corr_cols].dropna()
corrs = valid_data.corr()['delay_rate'].drop('delay_rate').sort_values(ascending=False)

print('Correlation with delay_rate:')
print('-' * 50)
for feat, val in corrs.items():
    marker = ' <-- LF' if 'load_factor' in feat else ''
    print('  {:<30} {:+.4f}{}'.format(feat, val, marker))

## 3. Data Availability and Train/Validation/Test Split

Both models are trained on the same data subset (rows where lag1, lag2, gradient, lag12, and load_factor_lag1 are all available) to ensure fair comparison.

In [ ]:
# Drop NaN and report data loss
required_cols = ['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient',
                 'delay_rate_lag12', 'load_factor_lag1']

n_before = len(df_filtered)
n_after_lags = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).shape[0]
n_after_lag12 = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient', 'delay_rate_lag12']).shape[0]
df_clean = df_filtered.dropna(subset=required_cols).copy()
n_final = len(df_clean)

print('Data availability:')
print('  After filtering:          {:,}'.format(n_before))
print('  After lag1/lag2/gradient:  {:,} (-{:,})'.format(n_after_lags, n_before - n_after_lags))
print('  After lag12:              {:,} (-{:,}, {:.1f}%)'.format(
    n_after_lag12, n_after_lags - n_after_lag12,
    (n_after_lags - n_after_lag12) / n_after_lags * 100))
print('  After load_factor_lag1:   {:,} (-{:,}, {:.1f}%)'.format(
    n_final, n_after_lag12 - n_final,
    (n_after_lag12 - n_final) / n_after_lag12 * 100 if n_after_lag12 > 0 else 0))

In [ ]:
# Train/validation/test split (same as 15a)
train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print('Train: {:,}  (years 2010-2017, 2023)'.format(train_mask.sum()))
print('Val:   {:,}  (years 2018, 2024)'.format(val_mask.sum()))
print('Test:  {:,}  (years 2019, 2025+)'.format(test_mask.sum()))

In [ ]:
# Define feature sets (no weather features for forecasting)
base_features = (airline_cols + route_cols
                 + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled'])
non_weather_features = ['delay_rate_gradient', 'n_public_holidays_total', 'pct_school_holiday']

# Baseline: forecasting with lag12 (from 15a)
features_baseline = base_features + ['delay_rate_lag12'] + non_weather_features

# With load_factor_lag1
features_with_lf = base_features + ['delay_rate_lag12', 'load_factor_lag1'] + non_weather_features

print('Baseline (Lag12):     {} features'.format(len(features_baseline)))
print('+ LF Lag1:            {} features'.format(len(features_with_lf)))

## 4. Model Comparison

Ridge, Random Forest, and XGBoost are trained on both feature sets using the same data and hyperparameters as notebook 15a. The incremental impact of adding `load_factor_lag1` is measured.

In [ ]:
def evaluate_model(df, features, train_mask, val_mask, test_mask, name):
    """Train and evaluate Ridge, RF, and XGBoost."""
    X_train = df.loc[train_mask, features].values
    X_val = df.loc[val_mask, features].values
    X_test = df.loc[test_mask, features].values

    y_train = df.loc[train_mask, 'delay_rate'].values
    y_val = df.loc[val_mask, 'delay_rate'].values
    y_test = df.loc[test_mask, 'delay_rate'].values

    y_train_clf = df.loc[train_mask, 'is_high_delay'].values
    y_val_clf = df.loc[val_mask, 'is_high_delay'].values
    y_test_clf = df.loc[test_mask, 'is_high_delay'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    results = {'name': name}

    # Ridge
    ridge = Ridge(alpha=100)
    ridge.fit(X_train_scaled, y_train)
    ridge_pred = ridge.predict(X_test_scaled)
    results['ridge_r2'] = r2_score(y_test, ridge_pred)
    results['ridge_rmse'] = np.sqrt(mean_squared_error(y_test, ridge_pred))

    # Random Forest Regression
    rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5,
                               random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    results['rf_r2'] = r2_score(y_test, rf_pred)
    results['rf_rmse'] = np.sqrt(mean_squared_error(y_test, rf_pred))
    results['rf_model'] = rf

    # XGBoost Classification
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                     min_child_weight=5, random_state=42, n_jobs=-1)
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_pred = xgb_clf.predict(X_test)
        results['xgb_f1'] = f1_score(y_test_clf, xgb_pred)
    else:
        results['xgb_f1'] = np.nan

    # Store predictions for route-level analysis
    results['ridge_pred'] = ridge_pred
    results['rf_pred'] = rf_pred
    results['y_test'] = y_test

    return results, features

In [ ]:
# Train and evaluate both variants
print('Training models...')
print()

results_baseline, _ = evaluate_model(
    df_clean, features_baseline, train_mask, val_mask, test_mask, 'Baseline (Lag12)')
results_lf, _ = evaluate_model(
    df_clean, features_with_lf, train_mask, val_mask, test_mask, '+ LF Lag1')

print('=' * 75)
print('FORECASTING MODEL COMPARISON')
print('=' * 75)
print()

header = '{:<20} {:>10} {:>12} {:>10} {:>12} {:>10}'.format(
    'Variant', 'Ridge R\u00b2', 'Ridge RMSE', 'RF R\u00b2', 'RF RMSE', 'XGB F1')
print(header)
print('-' * 75)

for r in [results_baseline, results_lf]:
    xgb_str = '{:.4f}'.format(r['xgb_f1']) if not np.isnan(r.get('xgb_f1', np.nan)) else 'N/A'
    print('{:<20} {:>10.4f} {:>12.4f} {:>10.4f} {:>12.4f} {:>10}'.format(
        r['name'], r['ridge_r2'], r['ridge_rmse'], r['rf_r2'], r['rf_rmse'], xgb_str))

print()
print('Delta (+ LF Lag1 - Baseline):')
d_ridge = results_lf['ridge_r2'] - results_baseline['ridge_r2']
d_rf = results_lf['rf_r2'] - results_baseline['rf_r2']
d_xgb = results_lf['xgb_f1'] - results_baseline['xgb_f1'] if HAS_XGB else 0
print('  Ridge R\u00b2:   {:+.4f}'.format(d_ridge))
print('  Ridge RMSE: {:+.4f}'.format(results_lf['ridge_rmse'] - results_baseline['ridge_rmse']))
print('  RF R\u00b2:      {:+.4f}'.format(d_rf))
print('  RF RMSE:    {:+.4f}'.format(results_lf['rf_rmse'] - results_baseline['rf_rmse']))
if HAS_XGB:
    print('  XGB F1:     {:+.4f}'.format(d_xgb))

## 5. Feature Importance

Random Forest feature importance reveals how `load_factor_lag1` ranks relative to established predictors in the forecasting feature set.

In [ ]:
# RF feature importance (with LF lag1)
rf_model = results_lf['rf_model']
importance_df = pd.DataFrame({
    'Feature': features_with_lf,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('RF Feature Importance (Forecasting + LF Lag1) - Top 15:')
print('-' * 55)
for rank, (_, row) in enumerate(importance_df.head(15).iterrows(), 1):
    marker = ' <-- LF' if 'load_factor' in row['Feature'] else ''
    print('  {:>2}. {:<35} {:.4f}{}'.format(rank, row['Feature'], row['Importance'], marker))

lf_rank = list(importance_df['Feature']).index('load_factor_lag1') + 1
lf_importance = importance_df[importance_df['Feature'] == 'load_factor_lag1']['Importance'].values[0]
print()
print('load_factor_lag1 rank: {} (importance: {:.4f})'.format(lf_rank, lf_importance))

In [ ]:
# Feature importance bar chart
fig, ax = plt.subplots(figsize=(10, 6))

top_features = importance_df.head(12)
y_pos = np.arange(len(top_features))
colors = ['#f39c12' if 'load_factor' in f else 'steelblue' for f in top_features['Feature']]

ax.barh(y_pos, top_features['Importance'], color=colors, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_features['Feature'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Feature Importance: Forecasting with Lag12 + LF Lag1')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 6. Route-Level Analysis

Route-level performance comparison reveals whether `load_factor_lag1` benefits specific routes or has a uniform impact across the network.

In [ ]:
# Route-level R\u00b2 comparison (Ridge)
df_test = df_clean[test_mask].copy()
df_test['baseline_pred'] = results_baseline['ridge_pred']
df_test['lf_pred'] = results_lf['ridge_pred']

print('Route-Level Performance (Ridge R\u00b2):')
print('=' * 70)
print('{:<25} {:>10} {:>10} {:>10}'.format('Route', 'Baseline', '+ LF Lag1', 'Delta'))
print('-' * 70)

route_results = []
for route in sorted(df_test['route'].unique()):
    rd = df_test[df_test['route'] == route]
    r2_bl = r2_score(rd['delay_rate'], rd['baseline_pred'])
    r2_lf = r2_score(rd['delay_rate'], rd['lf_pred'])
    delta = r2_lf - r2_bl
    print('{:<25} {:>10.4f} {:>10.4f} {:>+10.4f}'.format(route, r2_bl, r2_lf, delta))
    route_results.append({'Route': route, 'Baseline': r2_bl, 'LF_Lag1': r2_lf, 'Delta': delta})

route_df = pd.DataFrame(route_results)
print('-' * 70)
overall_bl = r2_score(df_test['delay_rate'], df_test['baseline_pred'])
overall_lf = r2_score(df_test['delay_rate'], df_test['lf_pred'])
print('{:<25} {:>10.4f} {:>10.4f} {:>+10.4f}'.format(
    'OVERALL', overall_bl, overall_lf, overall_lf - overall_bl))

improved = (route_df['Delta'] > 0).sum()
print()
print('{}/{} routes improved'.format(improved, len(route_df)))

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))

route_df_sorted = route_df.sort_values('Delta')
x = np.arange(len(route_df_sorted))
colors = ['#27ae60' if d > 0 else '#e74c3c' for d in route_df_sorted['Delta']]

ax.bar(x, route_df_sorted['Delta'], color=colors, alpha=0.85)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xticks(x)
route_labels = [r.replace('_', ' \u2192 ') for r in route_df_sorted['Route']]
ax.set_xticklabels(route_labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('\u0394 R\u00b2 (Ridge)')
ax.set_title('Route-Level Impact of load_factor_lag1 on Forecasting')
ax.grid(True, alpha=0.3, axis='y')

ax.text(0.98, 0.98, '{}/{} routes improved'.format(improved, len(route_df)),
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 7. Summary and Observations

In [ ]:
print('=' * 75)
print('CONCLUSIONS: FORECASTING WITH LOAD FACTOR LAG1')
print('=' * 75)
print()

threshold = 0.005

print('1. PERFORMANCE IMPACT:')
print('   Ridge R\u00b2:   {:.4f} -> {:.4f} ({:+.4f})'.format(
    results_baseline['ridge_r2'], results_lf['ridge_r2'], d_ridge))
print('   RF R\u00b2:      {:.4f} -> {:.4f} ({:+.4f})'.format(
    results_baseline['rf_r2'], results_lf['rf_r2'], d_rf))
if HAS_XGB:
    print('   XGBoost F1: {:.4f} -> {:.4f} ({:+.4f})'.format(
        results_baseline['xgb_f1'], results_lf['xgb_f1'], d_xgb))
print()

print('2. FEATURE IMPORTANCE:')
print('   load_factor_lag1 rank: {} (importance: {:.4f})'.format(lf_rank, lf_importance))
print()

print('3. ROUTE-LEVEL:')
print('   {}/{} routes improved'.format(improved, len(route_df)))
print()

print('4. COMPARISON WITH NOWCASTING (notebook 17b):')
print('   Nowcasting NN + LF Raw: F1 +0.029 (0.743 -> 0.772)')
print('   Forecasting + LF Lag1:  Ridge R\u00b2 {:+.4f}, RF R\u00b2 {:+.4f}'.format(d_ridge, d_rf))
print()

# Verdict
improvements = sum([d_ridge > threshold, d_rf > threshold,
                    d_xgb > threshold if HAS_XGB else False])

if improvements >= 2:
    verdict = 'RECOMMENDED - load_factor_lag1 improves {} of 3 models by >{}'.format(
        improvements, threshold)
elif improvements == 1:
    verdict = 'MARGINAL - load_factor_lag1 improves only 1 model meaningfully'
else:
    all_positive = all([d_ridge > 0, d_rf > 0, d_xgb > 0 if HAS_XGB else True])
    if all_positive:
        verdict = 'MARGINAL - all models improve but below threshold ({})'.format(threshold)
    else:
        verdict = 'NOT RECOMMENDED - load_factor_lag1 does not consistently improve forecasting'


### Observations

- Unfortunately, `load_factor_lag1` does not improve forecasting using regression models
- XGBoost classification model moderately benefit from it.
- However, the caveat is: the current load factor data are industry-aggregate (one value per month for all airlines). If route-level load factor data is available in the futute, it might provide a stronger signal.

## 8. Next Step

The hyperparameters have not been tuned for the forecasting model. The next notebook will find the optimal hyperparameters.

##